# 스트리밍 도중 툴 호출 받기 — Claude Code 스타일

Claude Code는 응답을 스트리밍으로 받다가 **응답이 끝나기 전에** 툴 사용 요청이 도착하면,
그 시점에 바로 "도구 실행 중..." UI를 띄웁니다. OpenAI에서도 같은 게 가능한지 검증합니다.

검증 포인트 3가지:
1. 텍스트와 툴 호출이 **하나의 SSE 스트림** 안에서 순서대로 도착하는가?
2. 툴 호출은 **응답 종료 전에**, 함수 이름부터 먼저 공개되는가? (UI를 일찍 그릴 수 있는가)
3. 툴 실행 후 이어지는 응답도 스트리밍되는가? (전 구간 스트리밍 에이전트 루프)

각 이벤트에 **경과 시간(초)** 을 찍어서, 한꺼번에 오는 게 아니라 실시간으로 흘러오는 것을 확인합니다.

In [1]:
import json
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = "gpt-5-nano"

TOOLS = [{
    "type": "function", "name": "get_weather",
    "description": "도시 하나의 현재 날씨 조회",
    "parameters": {"type": "object", "properties": {"city": {"type": "string"}},
                   "required": ["city"], "additionalProperties": False},
    "strict": True,
}]

def get_weather(city: str) -> dict:
    mock = {"서울": (29, "맑음"), "부산": (27, "구름 조금"), "제주": (26, "비")}
    temp, cond = mock.get(city, (25, "정보 없음"))
    return {"city": city, "temp_c": temp, "condition": cond}

## 1. 스트림 하나를 해부해보기 — 텍스트 → 툴 호출이 한 스트림에

`stream=True`로 요청하고, 도착하는 모든 이벤트에 타임스탬프를 찍습니다.
"확인 전에 뭘 할지 먼저 말해"라고 유도해서 텍스트가 먼저 나오게 합니다.

In [2]:
stream = client.responses.create(
    model=MODEL,
    input="서울 날씨 확인해줘. 확인하기 전에 뭘 할건지 한 문장으로 먼저 말해.",
    tools=TOOLS,
    stream=True,
)

t0 = time.perf_counter()
ts = lambda: f"[{time.perf_counter() - t0:6.2f}s]"
text_buf = ""

for event in stream:
    et = event.type
    if et == "response.output_item.added" and event.item.type == "function_call":
        print(f"{ts()} ⚡ 툴 호출 아이템 시작 — 이 시점에 이미 이름을 안다: {event.item.name}")
    elif et == "response.output_text.delta":
        if not text_buf:
            print(f"{ts()} 💬 텍스트 delta 흘러오기 시작...")
        text_buf += event.delta
    elif et == "response.output_text.done":
        print(f"{ts()} 💬 텍스트 아이템 완료: {text_buf!r}")
        text_buf = ""
    elif et == "response.function_call_arguments.delta":
        print(f"{ts()} ⚡ 인자 조각 도착: {event.delta!r}")
    elif et == "response.function_call_arguments.done":
        print(f"{ts()} ⚡ 인자 완성: {event.arguments}")
    elif et == "response.completed":
        print(f"{ts()} ✅ 스트림(응답) 종료 — 아이템 순서: "
              + " → ".join(i.type for i in event.response.output))

[  0.92s] 💬 텍스트 delta 흘러오기 시작...
[  1.23s] 💬 텍스트 아이템 완료: '서울의 현재 날씨 정보를 조회해 확인하겠습니다.'
[  1.23s] ⚡ 툴 호출 아이템 시작 — 이 시점에 이미 이름을 안다: get_weather
[  1.23s] ⚡ 인자 조각 도착: '{"'
[  1.27s] ⚡ 인자 조각 도착: 'city'
[  1.27s] ⚡ 인자 조각 도착: '":"'
[  1.28s] ⚡ 인자 조각 도착: '서울'
[  1.28s] ⚡ 인자 조각 도착: '"}'
[  1.57s] ⚡ 인자 완성: {"city":"서울"}
[  1.57s] ✅ 스트림(응답) 종료 — 아이템 순서: message → function_call


**읽는 법**: 텍스트 delta가 실시간으로 흐르고 → 텍스트 아이템이 닫힌 직후 → **같은 스트림에서** 툴 호출 아이템이
이름과 함께 열리고 → 인자가 조각조각 흘러온 뒤 → 응답이 종료됩니다.

즉 **툴 요청은 응답 종료 전에, 스트리밍 중간에 도착**합니다 — Claude Code가 툴 실행 표시를 즉시 띄울 수 있는 것과 같은 구조입니다.
(단, 텍스트 delta 사이에 툴 호출이 끼어드는 건 아니고, 블록 단위 순차 — 이것도 Claude와 동일)

## 2. 전 구간 스트리밍 에이전트 루프 — Claude Code의 실제 동작 재현

스트림 도중 툴 호출을 감지하면 → 즉시 "실행 중" 표시 → 로컬 함수 실행 →
결과를 넣고 다음 요청도 스트리밍 → 최종 답변이 실시간으로 흘러옵니다.

In [3]:
input_list = [{"role": "user", "content": "서울이랑 제주 날씨 비교해줘. 시작 전에 계획을 한 문장으로 말해."}]

round_no = 0
while True:
    round_no += 1
    print(f"\n───────── 스트림 #{round_no} ─────────")
    stream = client.responses.create(model=MODEL, input=input_list, tools=TOOLS, stream=True)

    final_response = None
    for event in stream:
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)          # 텍스트 실시간 표시
        elif event.type == "response.output_item.added" and event.item.type == "function_call":
            print(f"\n  ⚡ [스트림 도중 감지] {event.item.name} 실행 준비...", flush=True)
        elif event.type == "response.completed":
            final_response = event.response

    input_list += final_response.output
    calls = [i for i in final_response.output if i.type == "function_call"]
    if not calls:
        print("\n\n✅ 루프 종료 — 모든 구간이 스트리밍으로 처리됨")
        break
    for call in calls:
        result = get_weather(**json.loads(call.arguments))
        print(f"  🔧 로컬 실행: {call.name}({call.arguments}) → {result}")
        input_list.append({"type": "function_call_output", "call_id": call.call_id,
                           "output": json.dumps(result, ensure_ascii=False)})


───────── 스트림 #1 ─────────
서울과 제주 각각의 현재 날씨를 조회한 뒤, 기온과 상태를 중심으로 간단히 비교하겠습니다.
  ⚡ [스트림 도중 감지] get_weather 실행 준비...

  ⚡ [스트림 도중 감지] get_weather 실행 준비...
  🔧 로컬 실행: get_weather({"city":"서울"}) → {'city': '서울', 'temp_c': 29, 'condition': '맑음'}
  🔧 로컬 실행: get_weather({"city":"제주"}) → {'city': '제주', 'temp_c': 26, 'condition': '비'}

───────── 스트림 #2 ─────────
현재 날씨 비교입니다.

- **서울**: 29°C, 맑음
- **제주**: 26°C, 비

비교하면 **서울이 제주보다 3°C 더 높고**, 날씨도 서울은 **맑은 반면** 제주는 **비가 내리고 있습니다**.  
외출하기에는 서울이 더 쾌적해 보이고, 제주는 우산이 필요합니다.

✅ 루프 종료 — 모든 구간이 스트리밍으로 처리됨


## 3. 병렬 멀티 호출도 스트림에 순차 도착

도시 3개를 물으면 `function_call` 아이템 3개가 **한 스트림 안에서 차례로** 흘러옵니다.
첫 호출의 인자가 완성되는 순간, 나머지가 아직 스트리밍 중이어도 첫 함수 실행을 시작할 수 있습니다.
(제미나이처럼 완성된 호출 목록이 마지막에 한 덩어리로 오는 방식과의 실질적 차이)

In [4]:
stream = client.responses.create(
    model=MODEL,
    input="서울, 부산, 제주 세 도시의 날씨를 지금 바로 각각 확인해줘.",
    tools=TOOLS,
    stream=True,
)

t0 = time.perf_counter()
ts = lambda: f"[{time.perf_counter() - t0:6.2f}s]"

for event in stream:
    if event.type == "response.output_item.added" and event.item.type == "function_call":
        print(f"{ts()} ⚡ 호출 시작: {event.item.name}")
    elif event.type == "response.function_call_arguments.done":
        print(f"{ts()}    인자 완성: {event.arguments}  ← 이 시점부터 이 함수는 실행 가능")
    elif event.type == "response.completed":
        print(f"{ts()} ✅ 스트림 종료 — "
              + " → ".join(i.type for i in event.response.output))

[  1.13s] ⚡ 호출 시작: get_weather
[  1.13s]    인자 완성: {"city":"서울"}  ← 이 시점부터 이 함수는 실행 가능
[  1.13s] ⚡ 호출 시작: get_weather
[  1.13s]    인자 완성: {"city":"부산"}  ← 이 시점부터 이 함수는 실행 가능
[  1.13s] ⚡ 호출 시작: get_weather
[  1.13s]    인자 완성: {"city":"제주"}  ← 이 시점부터 이 함수는 실행 가능
[  1.23s] ✅ 스트림 종료 — function_call → function_call → function_call


## 정리

| 검증 포인트 | 결과 |
|---|---|
| 텍스트와 툴 호출이 한 스트림에 | ✅ `message` 아이템 → `function_call` 아이템 순서로 같은 SSE 스트림에 도착 |
| 툴 요청이 응답 종료 전에 도착 | ✅ `output_item.added`에서 함수 이름 즉시 공개 → UI를 일찍 그릴 수 있음 |
| 인자도 스트리밍 | ✅ `function_call_arguments.delta`로 조각조각 (완성 전엔 JSON 파싱 불가 주의) |
| 툴 실행 후 구간도 스트리밍 | ✅ 요청 단위로 스트림이 나뉘지만 모든 구간 스트리밍 가능 |
| 텍스트 delta "사이에" 툴 호출이 끼어듦 | ❌ 블록(아이템) 단위 순차 — 텍스트 아이템이 닫힌 뒤 툴 아이템 시작. **Claude도 동일** |

**결론**: Claude Code가 하는 "스트리밍 도중 툴 요청 수신 → 즉시 실행 표시"는 OpenAI Responses API로도 동일하게 구현 가능합니다.
Claude(`content_block_start`/`input_json_delta`)와 OpenAI(`output_item.added`/`function_call_arguments.delta`)는
이벤트 이름만 다를 뿐 같은 설계이고, 완성된 호출을 응답 끝에 한꺼번에 주는 제미나이 방식과 구분됩니다.